# 02 · Múltiplas ferramentas e a evolução do contexto

Estendemos o agente de uma para três ferramentas. O ponto central do exercício é observar que o
laço de controle permanece inalterado: a seleção entre ferramentas, bem como a decisão de
encadeá-las, é integralmente delegada ao modelo. Examinamos ainda o crescimento do histórico
(`messages`) ao longo da execução — o substrato que permite ao agente condicionar cada decisão
nas observações acumuladas.


## Dependências

In [ ]:
!pip install -q openai

## Configuração do cliente

Empregamos a biblioteca `openai` apontando para o endpoint **gratuito da NVIDIA**
(`https://integrate.api.nvidia.com/v1`), que expõe uma interface compatível com a API da OpenAI.
A célula seguinte concentra os parâmetros de acesso: endpoint, modelo e credencial.

Para obter a credencial: criar conta em https://build.nvidia.com e gerar uma API key (prefixo
`nvapi-`). No Google Colab, a chave pode ser guardada uma única vez em *Secrets* (ícone de chave na
barra lateral) com o nome `NVIDIA_API_KEY`; a célula a seguir a detecta automaticamente. Para
utilizar outro modelo, basta substituir a variável `MODEL` por outro identificador do catálogo
(https://build.nvidia.com/models) — nenhum outro ponto do código precisa mudar.


In [ ]:
import os
import json
from getpass import getpass
from datetime import datetime
from openai import OpenAI

def _obter_api_key():
    try:                                  # 1) Colab Secrets (icone de chave a esquerda)
        from google.colab import userdata
        chave = userdata.get("NVIDIA_API_KEY")
        if chave:
            return chave
    except Exception:
        pass
    return os.environ.get("NVIDIA_API_KEY") or getpass("API key (nvapi-...): ")  # 2) env var ou 3) manual

# Parametros de backend (alterar para usar outro provedor)
BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL    = "meta/llama-3.1-8b-instruct"
API_KEY  = _obter_api_key()

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("Cliente configurado:", BASE_URL, "| modelo:", MODEL)


## Definição das ferramentas

In [ ]:
def get_current_date() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"


def get_weather(city: str) -> str:
    # Implementacao simulada; substituir por chamada a uma API de clima
    return f"O tempo em {city} e 22C, parcialmente nublado."


TOOL_FUNCTIONS = {
    "get_current_date": get_current_date,
    "calculate": calculate,
    "get_weather": get_weather,
}


## Especificações das ferramentas

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_current_date",
        "description": "Retorna a data e hora atuais.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Avalia uma expressao matematica e retorna o resultado.",
        "parameters": {"type": "object",
            "properties": {"expression": {"type": "string", "description": "ex.: '847 * 0.15'"}},
            "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Retorna o clima atual de uma cidade.",
        "parameters": {"type": "object",
            "properties": {"city": {"type": "string", "description": "Nome da cidade"}},
            "required": ["city"]}}},
]


## O laço, instrumentado

O laço reproduz o do notebook anterior, acrescido da impressão do tamanho do histórico a cada
iteração. O acúmulo progressivo de mensagens — solicitações do modelo, invocações de ferramenta e
observações — é o que dota o agente da capacidade de raciocinar de forma incremental sobre uma
tarefa composta.


In [ ]:
def run_agent(task: str, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": "Voce e um assistente util. Use ferramentas quando precisar. Quando tiver a resposta final, responda sem chamar ferramentas."},
        {"role": "user", "content": task},
    ]

    turno = 0
    while True:
        turno += 1
        if verbose:
            print(f"[iteracao {turno} | {len(messages)} mensagens no historico]")

        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto",
        )
        message = response.choices[0].message
        if message.tool_calls and len(message.tool_calls) > 1:
            message.tool_calls = message.tool_calls[:1]  # este endpoint aceita 1 tool call por turno
        messages.append(message)

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            if verbose:
                print(f"   -> {name}({args})")
            fn = TOOL_FUNCTIONS.get(name)
            result = fn(**args) if fn else f"Ferramenta desconhecida: {name}"
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(result)})


## Execução

Uma tarefa que requer as três ferramentas. Observe a expansão do histórico e o encadeamento das
invocações que precede a resposta final.


In [ ]:
task = "Qual e a data de hoje? Quanto e 15% de 847? E como esta o tempo em Toquio?"
print("Tarefa:", task, "\n")
print("\nResposta:\n", run_agent(task))


## Exercício

Acrescente uma quarta ferramenta, `tamanho_do_texto(texto)`, que retorne o número de caracteres
de uma cadeia. São necessários dois registros: a função em `TOOL_FUNCTIONS` e a especificação
correspondente em `TOOLS`. A célula abaixo apresenta uma solução de referência; reescreva-a como
exercício antes de consultá-la.


In [ ]:
def tamanho_do_texto(texto: str) -> str:
    return str(len(texto))

TOOL_FUNCTIONS["tamanho_do_texto"] = tamanho_do_texto

TOOLS.append({"type": "function", "function": {
    "name": "tamanho_do_texto",
    "description": "Retorna o numero de caracteres de um texto.",
    "parameters": {"type": "object",
        "properties": {"texto": {"type": "string", "description": "O texto a medir"}},
        "required": ["texto"]}}})

print(run_agent("Quantos caracteres tem a palavra 'paralelepipedo'?"))


---
Continuação: `03_trocando_modelo.ipynb` — substituição do modelo subjacente e suas implicações,
incluindo a dependência do agente em relação ao suporte a *function calling*.
